## Setup

In [2]:
# =============================================================================
# Wild Boar Functional Connectivity Pipeline
# Methods: Graph Theory + Circuitscape (Pinch-points) + Least-Cost Corridors
# Based on: Urbina et al. (in-review) approach as applied in Rothirsch Mittelland
# =============================================================================
# REQUIREMENTS:
#   R packages: terra, sf, igraph, gdistance, dplyr, ggplot2
#   External:   Circuitscape via Julia (julia + Circuitscape.jl installed)
#   Input 1:    Resistance raster  -> ../data/processed/Resistance_Keeley_Aargau_25m.tif
#   Input 2:    Core habitat patches (Kerngebiete) -> set path in CONFIG below
# =============================================================================

# --------------------------------------------- 0. PACKAGES & CONFIG ----------

install.packages(c("terra", "sf", "igraph", "gdistance", "spdep", "dplyr", "ggplot2"))

library(terra)      # raster/vector operations
library(sf)         # vector operations
library(igraph)     # landscape graph
library(gdistance)  # least-cost paths & cumulative costs
library(dplyr)
library(ggplot2)

# ── USER CONFIG ───────────────────────────────────────────────────────────────
RESISTANCE_PATH  <- "../data/processed/Resistance_Keeley_Wildboar_Zurich_25m.tif"
CORE_PATCHES_PATH <- "../data/raw/Daten_Bericht_VetAmt/Daten_Bericht_VetAmt/Daten/shapefiles/Potenzial_Wildschwein_Kerneinstaende_ZH_2024_WILMA/Potenzial_Wildschwein_Kerneinstaende_ZH_2024_WILMA.shp"
OUT_DIR          <- "../data/processed/connectivity/"
MAX_DIST_KM      <- 30        # max Euclidean distance for graph edges (km)
TILE_SIZE_CELLS  <- 400       # Circuitscape tile size (pixels); adjust to RAM
JULIA_PATH       <- "C:/Users/Lukas/AppData/Local/Programs/Julia-1.12.6/bin/julia.exe"   # path to julia executable, e.g. "/usr/local/bin/julia"
# ─────────────────────────────────────────────────────────────────────────────

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(OUT_DIR, "circuitscape_tiles"), recursive = TRUE, showWarnings = FALSE)




Installing packages into ‘C:/Users/Lukas/AppData/Local/R/win-library/4.5’
(as ‘lib’ is unspecified)


trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/terra_1.9-11.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/sf_1.1-0.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/igraph_2.3.0.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/gdistance_1.6.5.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/spdep_1.4-2.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/dplyr_1.2.1.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/ggplot2_4.0.3.zip'


package ‘terra’ successfully unpacked and MD5 sums checked


package ‘sf’ successfully unpacked and MD5 sums checked


package ‘igraph’ successfully unpacked and MD5 sums checked
package ‘gdistance’ successfully unpacked and MD5 sums checked
package ‘spdep’ successfully unpacked and MD5 sums checked
package ‘dplyr’ successfully unpacked and MD5 sums checked


...

Warning message:
package ‘igraph’ was built under R version 4.5.3 


Loading required package: raster
Loading required package: sp
Loading required package: Matrix

Attaching package: ‘gdistance’

The following object is masked from ‘package:igraph’:

    normalize



Warning messages:
1: package ‘gdistance’ was built under R version 4.5.3 
2: package ‘raster’ was built under R version 4.5.2 
3: package ‘sp’ was built under R version 4.5.2 



Attaching package: ‘dplyr’

The following objects are masked from ‘package:raster’:

    intersect, select, union

The following objects are masked from ‘package:igraph’:

    as_data_frame, groups, union

The following objects are masked from ‘package:terra’:

    intersect, union

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Warning message:
package ‘dplyr’ was built under R version 4.5.3 
Warning message:
package ‘ggplot2’ was built under R version 4.5.3 


In [3]:
# --------------------------------------------- 1. LOAD INPUTS ----------------

cat("── 1. Loading inputs ──\n")
resistance <- rast(RESISTANCE_PATH)
cores_sf   <- st_read(CORE_PATCHES_PATH, quiet = TRUE)

# Reproject cores to match resistance raster CRS if needed
cores_sf <- st_transform(cores_sf, crs = crs(resistance))

# Add unique ID column if not present
if (!"patch_id" %in% names(cores_sf)) {
  cores_sf$patch_id <- seq_len(nrow(cores_sf))
}

# Compute centroids for graph nodes
centroids <- st_centroid(cores_sf)
coords    <- st_coordinates(centroids)   # matrix [x, y]

cat(sprintf("  Resistance raster: %d rows x %d cols, res = %.1f m\n",
            nrow(resistance), ncol(resistance), res(resistance)[1]))
cat(sprintf("  Core patches loaded: %d patches\n", nrow(cores_sf)))




── 1. Loading inputs ──


Warning message:
st_centroid assumes attributes are constant over geometries 


  Resistance raster: 2378 rows x 1906 cols, res = 25.0 m
  Core patches loaded: 3605 patches


## Landscape diagram

In [5]:
# --------------------------------------------- 2. LANDSCAPE GRAPH ------------
# Planar graph between core patch centroids with max Euclidean distance of 30 km
# Equivalent to Graphab planar graph approach (Foltête et al. 2021)

cat("\n── 2. Building landscape graph ──\n")

# Euclidean distance matrix between centroids (in metres)
dist_mat <- as.matrix(dist(coords))

# Build planar graph: connect each node to its nearest neighbours within MAX_DIST_KM
# using a Delaunay triangulation to avoid edge crossings (planar approximation)
max_dist_m <- MAX_DIST_KM * 1000

# Delaunay triangulation via sf for planarity
library(spdep)   # for tri2nb
nb <- tri2nb(coords)   # Delaunay triangulation neighbour list

# Convert to edge list, filter by max distance
edges <- do.call(rbind, lapply(seq_along(nb), function(i) {
  j_vec <- nb[[i]]
  j_vec <- j_vec[j_vec > i]  # upper triangle only
  if (length(j_vec) == 0) return(NULL)
  data.frame(from = i, to = j_vec,
             dist_m = dist_mat[i, j_vec])
}))
edges <- edges[edges$dist_m <= max_dist_m, ]

cat(sprintf("  Graph edges after distance filter: %d\n", nrow(edges)))

# Create igraph object
g <- graph_from_data_frame(edges, directed = FALSE,
                           vertices = data.frame(id = seq_len(nrow(cores_sf)),
                                                 x  = coords[, 1],
                                                 y  = coords[, 2]))

# Save graph edge list (node pairs used in steps 3 & 4)
write.csv(edges, file.path(OUT_DIR, "graph_edges.csv"), row.names = FALSE)

# Save graph plot
png(file.path(OUT_DIR, "landscape_graph.png"), width = 1800, height = 1600, res = 200)
plot(st_geometry(cores_sf), col = "#2d6a4f55", border = "#2d6a4f",
     main = "Landscape Graph – Wild Boar Core Patches")
for (i in seq_len(nrow(edges))) {
  lines(rbind(coords[edges$from[i], ], coords[edges$to[i], ]),
        col = "#e76f51", lwd = 1.2)
}
points(coords, pch = 21, bg = "#2d6a4f", col = "white", cex = 1.5)
dev.off()
cat("  Graph saved to landscape_graph.png\n")





── 2. Building landscape graph ──
  Graph edges after distance filter: 10785
  Graph saved to landscape_graph.png


In [15]:
# --------------------------------------------- 2. FILTER HABITATS -----------
cat("── 2. Filtering habitats by area ──\n")

# Define minimum area threshold (2 Hectares = 20,000 m2)
# 10 ha is an ideal minimum for a wild boar daytime resting core
min_area_m2 <- 20000 

# 1. Calculate area for each patch
cores_sf$area_m2 <- as.numeric(st_area(cores_sf))

# 2. Filter for patches meeting the threshold
cores_filtered_sf <- cores_sf %>%
  filter(area_m2 >= min_area_m2)

# Update centroids and coordinates to match the filtered set
centroids <- st_centroid(cores_filtered_sf)
coords    <- st_coordinates(centroids)

cat(sprintf("  Patches remaining after area filter: %d (Dropped %d small patches)\n", 
            nrow(cores_filtered_sf), 
            nrow(cores_sf) - nrow(cores_filtered_sf)))

# --------------------------------------------- 3. SAVE FILTERED HABITATS ----
cat("── 3. Saving to processed folder ──\n")

# Define the output path based on your project structure
out_path <- "C:/ZHAW/6.Semester/BA/BA-wild-boar-connectivity-modeling/data/processed/Kerneinstaende_filtered.shp"

# Write the shapefile (append = FALSE ensures it overwrites older versions)
st_write(cores_filtered_sf, out_path, append = FALSE)

cat("  Successfully saved filtered core habitats to:\n", out_path, "\n")

── 2. Filtering habitats by area ──


Warning message:
st_centroid assumes attributes are constant over geometries 


  Patches remaining after area filter: 121 (Dropped 3484 small patches)
── 3. Saving to processed folder ──
Writing layer `Kerneinstaende_filtered' to data source 
  `C:/ZHAW/6.Semester/BA/BA-wild-boar-connectivity-modeling/data/processed/Kerneinstaende_filtered.shp' using driver `ESRI Shapefile'
Writing 121 features with 4 fields and geometry type Polygon.
  Successfully saved filtered core habitats to:
 C:/ZHAW/6.Semester/BA/BA-wild-boar-connectivity-modeling/data/processed/Kerneinstaende_filtered.shp 


In [16]:
# --------------------------------------------- 2. LANDSCAPE GRAPH ------------
# Planar graph between core patch centroids with max Euclidean distance of 30 km
# Equivalent to Graphab planar graph approach (Foltête et al. 2021)

cat("\n── 2. Building landscape graph ──\n")

# Euclidean distance matrix between centroids (in metres)
dist_mat <- as.matrix(dist(coords))

# Build planar graph: connect each node to its nearest neighbours within MAX_DIST_KM
# using a Delaunay triangulation to avoid edge crossings (planar approximation)
max_dist_m <- MAX_DIST_KM * 1000

# Delaunay triangulation via sf for planarity
library(spdep)   # for tri2nb
nb <- tri2nb(coords)   # Delaunay triangulation neighbour list

# Convert to edge list, filter by max distance
edges <- do.call(rbind, lapply(seq_along(nb), function(i) {
  j_vec <- nb[[i]]
  j_vec <- j_vec[j_vec > i]  # upper triangle only
  if (length(j_vec) == 0) return(NULL)
  data.frame(from = i, to = j_vec,
             dist_m = dist_mat[i, j_vec])
}))
edges <- edges[edges$dist_m <= max_dist_m, ]

cat(sprintf("  Graph edges after distance filter: %d\n", nrow(edges)))

# Create igraph object
g <- graph_from_data_frame(edges, directed = FALSE,
                           vertices = data.frame(id = seq_len(nrow(cores_filtered_sf)),
                                                 x  = coords[, 1],
                                                 y  = coords[, 2]))

# Save graph edge list (node pairs used in steps 3 & 4)
write.csv(edges, file.path(OUT_DIR, "graph_edges_filtered.csv"), row.names = FALSE)

# Save graph plot
png(file.path(OUT_DIR, "landscape_graph_filtered.png"), width = 1800, height = 1600, res = 200)
plot(st_geometry(cores_filtered_sf), col = "#2d6a4f55", border = "#2d6a4f",
     main = "Landscape Graph – Wild Boar Core Patches Filtered")
for (i in seq_len(nrow(edges))) {
  lines(rbind(coords[edges$from[i], ], coords[edges$to[i], ]),
        col = "#e76f51", lwd = 1.2)
}
points(coords, pch = 21, bg = "#2d6a4f", col = "white", cex = 1.5)
dev.off()
cat("  Graph saved to landscape_graph_filtered.png\n")





── 2. Building landscape graph ──
  Graph edges after distance filter: 346
  Graph saved to landscape_graph_filtered.png


## Circuitscape Pinch-Points

In [21]:
cat("Installiere Circuitscape in Julia (das kann 1-2 Minuten dauern)...\n")

# Wir senden den Pkg.add() Befehl an Julia
cmd_install <- sprintf('%s -e "using Pkg; Pkg.add(\\"Circuitscape\\")"', JULIA_PATH)

# Ausführen - diesmal lassen wir den Output anzeigen, damit wir den Fortschritt sehen
system(cmd_install, ignore.stdout = FALSE, ignore.stderr = FALSE)

cat("\nInstallation abgeschlossen! Du kannst jetzt deinen Tile-Code neu starten.\n")

Installiere Circuitscape in Julia (das kann 1-2 Minuten dauern)...
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\Lukas\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\Lukas\.julia\environments\v1.12\Manifest.toml`

Installation abgeschlossen! Du kannst jetzt deinen Tile-Code neu starten.


In [27]:
 # =============================================================================
# CIRCUITSCAPE — One-to-all current flow between core habitat patches
#
# Method: Circuitscape 5 (Julia) in ONE-TO-ALL mode.
#   Each core patch acts as a current source in turn; all other nodes are
#   grounded. The cumulative current map (sum across all runs) identifies
#   pinch-points — pixels that current is forced through regardless of which
#   patch pair is considered.
#
# Why ONE-TO-ALL, not ADVANCED:
#  ADVANCED mode requires separate source_file and ground_file keys.
#   Omitting ground_file causes the Julia crash you saw:
#     "opening file (Browse for a ground point file)"
#   ONE-TO-ALL uses a single point_file for both roles — simpler and correct.
#
# Why NOT PAIRWISE:
#   O(n²) runtime. With 121 cores that is 7260 pairs. Use pairwise only if
#   you need per-pair current maps. For a cumulative pinch-point map,
#   one-to-all gives the same result at O(n) cost.
#
# Node file format for one-to-all (and pairwise):
#   Plain text, NO header, columns: node_id  x  y
#   Node IDs must be positive integers.
# =============================================================================

pacman::p_load(terra, sf, dplyr)

# -----------------------------------------------------------------------------
# CONFIG — adjust before running
# -----------------------------------------------------------------------------
RESISTANCE_TIF  <- "../data/processed/Resistance_Keeley_Wildboar_Zurich_25m.tif"
CORES_SHP       <- "../data/processed/Kerneinstaende_filtered.shp"
OUT_DIR         <- "../data/processed/circuitscape/"
JULIA_PATH      <- "julia"   # or full path e.g. "C:/Users/Lukas/AppData/Local/Programs/Julia-1.9.4/bin/julia.exe"

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

# =============================================================================
# 1. PREPARE RESISTANCE RASTER
#
# Circuitscape requires:
#   - ESRI ASCII grid (.asc)
#   - No CRS embedded (it ignores it — alignment is purely by extent/res)
#   - NoData = -9999
#   - Positive values only (resistance, not conductance)
# =============================================================================
cat("── 1. Preparing resistance raster ──\n")

resistance <- rast(RESISTANCE_TIF)

# Confirm no zero or negative values — Circuitscape will crash or give
# nonsense results with resistance = 0
min_val <- global(resistance, "min", na.rm = TRUE)[[1]]
if (min_val <= 0) {
  cat(sprintf("  WARNING: resistance contains values <= 0 (min = %.4f). Clamping to 0.01\n", min_val))
  resistance <- clamp(resistance, lower = 0.01, upper = Inf)
}

asc_path <- file.path(OUT_DIR, "resistance.asc")
writeRaster(resistance, asc_path,
            filetype = "AAIGrid",
            overwrite = TRUE,
            NAflag    = -9999)

cat(sprintf("  Saved: %s\n", asc_path))
cat(sprintf("  Dimensions: %d rows × %d cols, res = %.0f m\n",
            nrow(resistance), ncol(resistance), res(resistance)[1]))

# =============================================================================
# 2. PREPARE FOCAL NODES FROM CORE PATCHES
#
# Circuitscape focal_node file format:
#   - Plain text, space-separated
#   - Columns: node_id  x_coord  y_coord
#   - One row per core patch (centroid used)
#   - Node IDs must be positive integers
#   - Coordinates must be in the same projection as the resistance raster
# =============================================================================
cat("── 2. Preparing focal nodes ──\n")

cores_sf <- st_read(CORES_SHP, quiet = TRUE) %>%
  st_transform(crs(resistance))

# --- Snap each patch to its lowest-resistance cell ---------------------------
# A raw centroid may land on a barrier pixel (road, clearing) inside the patch,
# which causes Circuitscape to silently drop the node.
# Instead: for each patch polygon, extract all resistance pixels that fall
# within it and pick the cell with the minimum resistance value as the node.

cores_vect <- vect(cores_sf)
snap_to_min_resistance <- function(patch_vect, r) {
  cells     <- cells(r, patch_vect)[, "cell"]
  if (length(cells) == 0) return(NULL)
  vals      <- r[cells][, 1]
  best_cell <- cells[which.min(vals)]
  xy        <- xyFromCell(r, best_cell)
  data.frame(x = xy[1], y = xy[2])
}

cat("  Snapping nodes to lowest-resistance cell within each patch...\n")
node_coords <- lapply(seq_len(nrow(cores_vect)), function(i) {
  result <- snap_to_min_resistance(cores_vect[i], resistance)
  if (is.null(result)) {
    cat(sprintf("  WARNING: patch %d has no valid pixels — skipped\n", i))
    return(NULL)
  }
  cbind(node_id = i, result)
})

node_coords <- do.call(rbind, Filter(Negate(is.null), node_coords))

# Circuitscape one-to-all node file: no header, columns = node_id x y
nodes_path <- file.path(OUT_DIR, "focal_nodes.txt")
write.table(node_coords, nodes_path,
            row.names = FALSE, col.names = FALSE, quote = FALSE, sep = " ")

cat(sprintf("  %d / %d core patches → focal nodes written to %s\n",
            nrow(node_coords), nrow(cores_sf), nodes_path))

── 1. Preparing resistance raster ──
  Saved: ../data/processed/circuitscape//resistance.asc
  Dimensions: 2378 rows × 1906 cols, res = 25 m
── 2. Preparing focal nodes ──
  Snapping nodes to lowest-resistance cell within each patch...
  121 / 121 core patches → focal nodes written to ../data/processed/circuitscape//focal_nodes.txt


In [ ]:
print(cores_sf)

resistance

Simple feature collection with 121 features and 4 fields
Geometry type: POLYGON
Dimension:     XY
Bounding box:  xmin: 2672359 ymin: 1224200 xmax: 2715747 ymax: 1272518
Projected CRS: CH1903+ / LV95
First 10 features:
    Id gridcode patch_id  area_m2                       geometry
1  201        1      201 41099.49 POLYGON ((2687250 1272375, ...
2  232        1      232 25823.96 POLYGON ((2688050 1271875, ...
3  281        1      281 24879.27 POLYGON ((2700425 1270775, ...
4  285        1      285 71287.12 POLYGON ((2700350 1270550, ...
5  306        1      306 46298.07 POLYGON ((2685825 1269800, ...
6  337        1      337 24704.94 POLYGON ((2678575 1269175, ...
7  379        1      379 37165.20 POLYGON ((2685150 1268850, ...
8  401        1      401 27613.31 POLYGON ((2686900 1268175, ...
9  418        1      418 41086.29 POLYGON ((2679325 1267850, ...
10 480        1      480 20534.78 POLYGON ((2674300 1266800, ...
class       : SpatRaster 
size        : 2378, 1906, 1  (nrow, ncol, nlyr)
resolution  : 25.00288, 24.99884  (x, y)
extent      : 2669245, 2716900, 1223896, 1283343  (xmin, xmax, ymin, ymax)
coord. ref. : CH1903+ / LV95 (EPSG:2056) 
source      : Resistance_Keeley_Wildboar_Zurich_25m.tif 
name        : resistance_zurich 
min value   :           1.00000 
max value   :          54.59815 

Simple feature collection with 121 features and 4 fields
Geometry type: POLYGON
Dimension:     XY
Bounding box:  xmin: 2672359 ymin: 1224200 xmax: 2715747 ymax: 1272518
Projected CRS: CH1903+ / LV95
First 10 features:
    Id gridcode patch_id  area_m2                       geometry
1  201        1      201 41099.49 POLYGON ((2687250 1272375, ...
2  232        1      232 25823.96 POLYGON ((2688050 1271875, ...
3  281        1      281 24879.27 POLYGON ((2700425 1270775, ...
4  285        1      285 71287.12 POLYGON ((2700350 1270550, ...
5  306        1      306 46298.07 POLYGON ((2685825 1269800, ...
6  337        1      337 24704.94 POLYGON ((2678575 1269175, ...
7  379        1      379 37165.20 POLYGON ((2685150 1268850, ...
8  401        1      401 27613.31 POLYGON ((2686900 1268175, ...
9  418        1      418 41086.29 POLYGON ((2679325 1267850, ...
10 480        1      480 20534.78 POLYGON ((2674300 1266800, ...


class       : SpatRaster 
size        : 2378, 1906, 1  (nrow, ncol, nlyr)
resolution  : 25.00288, 24.99884  (x, y)
extent      : 2669245, 2716900, 1223896, 1283343  (xmin, xmax, ymin, ymax)
coord. ref. : CH1903+ / LV95 (EPSG:2056) 
source      : Resistance_Keeley_Wildboar_Zurich_25m.tif 
name        : resistance_zurich 
min value   :           1.00000 
max value   :          54.59815 

In [ ]:


# =============================================================================
# 3. WRITE .ini CONFIGURATION
#
# Using ADVANCED mode (one-to-all):
#   Each core acts as a current source in turn; all others act as grounds.
#   Cumulative current map = sum over all source iterations.
#   This is the standard approach for multi-patch connectivity mapping.
#
# Alternative — PAIRWISE mode:
#   Runs every pair explicitly. More precise, but O(n²) runtime.
#   Use if you have < ~30 cores and want pair-specific corridor maps.
#   Change scenario = pairwise below and it will work with the same node file.
# =============================================================================
cat("── 3. Writing Circuitscape .ini ──\n")

output_base <- file.path(OUT_DIR, "cs_output")

# Use forward slashes and normalised absolute paths — Circuitscape on Windows
# chokes on backslashes and relative paths inside the .ini
to_cs_path <- function(p)
  normalizePath(p, winslash = "/", mustWork = FALSE)

ini_text <- sprintf(
'[circuitscape options]
data_type                 = raster
scenario                  = one-to-all
write_cur_maps            = false
write_cum_cur_map_only    = true
write_volt_maps           = false
log_level                 = INFO
print_timings             = true

[Habitat raster or graph]
habitat_file              = %s
habitat_map_is_resistances = true

[Options for pairwise and one-to-all and all-to-one modes]
point_file                = %s
use_included_pairs        = false

[Output options]
output_file               = %s
',
to_cs_path(asc_path),
to_cs_path(nodes_path),
to_cs_path(paste0(output_base, ".out"))
)

ini_path <- file.path(OUT_DIR, "circuitscape.ini")
writeLines(ini_text, ini_path)
cat(sprintf("  Written: %s\n", ini_path))

# =============================================================================
# 4. RUN CIRCUITSCAPE VIA JULIA
# =============================================================================
cat("── 4. Running Circuitscape ──\n")
cat("  This may take several minutes depending on raster size and core count.\n")

# --project=@v1.9 is unnecessary and can cause version mismatches — removed
cmd <- sprintf(
  '%s -e "using Circuitscape; compute(\\"%s\\")"',
  JULIA_PATH,
  to_cs_path(ini_path)
)

cat(sprintf("  Command: %s\n\n", cmd))
exit_code <- system(cmd, ignore.stdout = FALSE, ignore.stderr = FALSE)

if (exit_code != 0) {
  stop(sprintf(
    "Circuitscape failed (exit code %d).\nCheck: %s_log.ini",
    exit_code, output_base
  ))
}

# =============================================================================
# 5. LOAD RESULT AND RECLASSIFY INTO DECILES
# =============================================================================
cat("── 5. Processing output ──\n")

# Circuitscape names the cumulative current map by replacing .out with
# _cum_curmap.asc next to the output file
curmap_path <- sub("\\.out$", "_cum_curmap.asc", paste0(output_base, ".out"))

if (!file.exists(curmap_path)) {
  stop(sprintf(
    "Expected output not found: %s\nCheck the Circuitscape log for errors.",
    curmap_path
  ))
}

current_density <- rast(curmap_path)

# Reattach CRS — Circuitscape strips projection info from ASC output
crs(current_density) <- crs(resistance)

# --- Diagnostics: check what came out of Circuitscape ------------------------
all_vals  <- as.numeric(values(current_density, na.rm = FALSE))
n_total   <- length(all_vals)
n_na      <- sum(is.na(all_vals))
n_zero    <- sum(all_vals == 0,  na.rm = TRUE)
n_nonzero <- sum(all_vals > 0,   na.rm = TRUE)
cat(sprintf("  Raster cells: %d total | %d NA | %d zero | %d non-zero\n",
            n_total, n_na, n_zero, n_nonzero))
cat(sprintf("  Value range (non-zero): [%.6f, %.4f]\n",
            min(all_vals[all_vals > 0], na.rm = TRUE),
            max(all_vals, na.rm = TRUE)))

if (n_nonzero == 0)
  stop("All output pixels are zero or NA. Check that focal_nodes.txt coordinates align with the resistance.asc extent.")

# --- Mask zeros to NA --------------------------------------------------------
# In one-to-all mode Circuitscape fills unreached pixels with 0.0, not NoData.
# Pixels truly outside the study area are already NA (from the -9999 nodata in
# the .asc). Zero-value pixels inside the study area are valid non-routed
# background cells — mask them to NA so they don't dominate the colour scale
# or the decile breaks.
current_density <- mask(current_density, current_density == 0,
                        maskvalue = TRUE, updatevalue = NA)

# Also align spatial extent/mask to the original resistance raster so any
# edge-of-tile artefacts from the ASC read are removed
current_density <- mask(current_density, resistance)

cat(sprintf("  Non-zero pixels retained: %d\n",
            sum(!is.na(values(current_density, na.rm = FALSE)))))

# --- Log-transform -----------------------------------------------------------
# Current density is extremely right-skewed. Log10 spreads the distribution
# so decile classification produces a readable map.
current_log <- log10(current_density)   # zeros are now NA, so log(0) never occurs

# --- Reclassify into deciles -------------------------------------------------
safe_decile_classify <- function(r) {
  vals   <- as.numeric(values(r, na.rm = TRUE))
  breaks <- unique(quantile(vals, probs = seq(0, 1, 0.1), na.rm = TRUE))

  if (length(breaks) < 3) {
    warning("Too few unique values for decile classification — returning raw raster.")
    return(r)
  }

  rcl <- cbind(
    as.numeric(breaks[-length(breaks)]),
    as.numeric(breaks[-1]),
    seq_len(length(breaks) - 1)
  )
  colnames(rcl) <- NULL

  classify(r, rcl = rcl, include.lowest = TRUE)
}

current_deciles <- safe_decile_classify(current_log)

# --- Save outputs ------------------------------------------------------------
writeRaster(current_density,
            file.path(OUT_DIR, "pinchpoint_current_density.tif"),
            overwrite = TRUE)
writeRaster(current_log,
            file.path(OUT_DIR, "pinchpoint_current_log10.tif"),
            overwrite = TRUE)
writeRaster(current_deciles,
            file.path(OUT_DIR, "pinchpoint_deciles.tif"),
            overwrite = TRUE)

# --- Plot --------------------------------------------------------------------
png(file.path(OUT_DIR, "pinchpoint_map.png"),
    width = 2400, height = 1200, res = 200)
par(mfrow = c(1, 2), mar = c(1, 1, 3, 3))

plot(current_log,
     col      = hcl.colors(100, "Lajolla"),
     main     = "Current density (log10)\nBackground = NA",
     axes     = FALSE,
     legend   = TRUE,
     colNA    = "#e8e8e8")

plot(current_deciles,
     col      = hcl.colors(10, "Lajolla"),
     main     = "Pinch-points (deciles of log10 current)\nDecile 10 = highest flow",
     axes     = FALSE,
     legend   = TRUE,
     colNA    = "#e8e8e8")

dev.off()

cat("\n── Done ──\n")
cat("  pinchpoint_current_density.tif  — raw Circuitscape output\n")
cat("  pinchpoint_current_log10.tif    — log10 transformed\n")
cat("  pinchpoint_deciles.tif          — decile-classified (use this in QGIS)\n")
cat("  pinchpoint_map.png\n")


── 1. Preparing resistance raster ──
  Saved: ../data/processed/circuitscape//resistance.asc
  Dimensions: 2378 rows × 1906 cols, res = 25 m
── 2. Preparing focal nodes ──
  Snapping nodes to lowest-resistance cell within each patch...
  121 / 121 core patches → focal nodes written to ../data/processed/circuitscape//focal_nodes.txt
── 3. Writing Circuitscape .ini ──
  Written: ../data/processed/circuitscape//circuitscape.ini
── 4. Running Circuitscape ──
  This may take several minutes depending on raster size and core count.
  Command: julia -e "using Circuitscape; compute(\"C:/ZHAW/6.Semester/BA/BA-wild-boar-connectivity-modeling/data/processed/circuitscape/circuitscape.ini\")"

[ Info: 2026-04-27 11:54:25 : Precision used: double
[ Info: 2026-04-27 11:54:26 : Reading maps
[ Info: 2026-04-27 11:54:35 : Resistance/Conductance map has 2773829 nodes
[ Info: 2026-04-27 11:54:49 : There are 2773829 points and 1 connected components
[ Info: 2026-04-27 11:54:50 : Solving point 1 of 121
[ In

Warning message:
In min(all_vals[all_vals > 0], na.rm = TRUE) :
  no non-missing arguments to min; returning Inf


: [1m[33mError[39m:[22m
[33m![39m All output pixels are zero or NA. Check that focal_nodes.txt coordinates align with the resistance.asc extent.

## Least Cost Path

In [25]:
# =============================================================================
# CUMULATIVE LEAST-COST CORRIDORS
#
# Method: For each edge in the habitat graph, compute the cost corridor as
#   corridor(i,j) = costDist(i) + costDist(j)
# The cumulative corridor surface is the running cell-wise minimum across
# all edges — areas that are part of many corridors get low values.
#
# Changes from previous version:
#   - Replaced gdistance (unmaintained) with terra::costDist() throughout
#   - No more sp::SpatialPoints / raster::raster conversions
#   - gc() removed from inner loop (terra manages memory internally)
#   - Added corridor width threshold to clip the final surface
#   - Decile inversion: decile 1 = best corridor, 10 = least suitable
# =============================================================================

pacman::p_load(terra, sf, dplyr, igraph, RANN)

# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------
RESISTANCE_TIF <- "../data/processed/Resistance_Keeley_Wildboar_Zurich_25m.tif"
CORES_SHP      <- "../data/processed/Kerneinstaende_filtered.shp"
OUT_DIR        <- "../data/processed/corridors/"

# Corridor width threshold: only keep pixels within this multiple of the
# least-cost path cost. 1.5 = 50% above the LCP = reasonably tight corridor.
# Increase to 2–3 for wider, more conservative corridors.
CORRIDOR_THRESHOLD <- 1.5

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

# =============================================================================
# 1. LOAD DATA
# =============================================================================
cat("── 1. Loading inputs ──\n")

resistance    <- rast(RESISTANCE_TIF)
cores_sf      <- st_read(CORES_SHP, quiet = TRUE) %>%
  st_transform(crs(resistance))

stopifnot("No core patches found" = nrow(cores_sf) > 0)

# Core centroids as a SpatVector (terra-native, no sp needed)
centroids  <- st_centroid(cores_sf)
cores_vect <- vect(centroids)

cat(sprintf("  %d core patches loaded\n", nrow(cores_sf)))

# Warn if any centroid falls on a barrier (NA or R_max) pixel
node_vals <- extract(resistance, cores_vect)[, 2]
R_max     <- exp(4)   # Keeley constant — adjust if you used a different C
bad       <- which(is.na(node_vals) | node_vals >= R_max * 0.95)
if (length(bad) > 0)
  warning(sprintf(
    "%d centroid(s) fall on barrier pixels (nodes %s). LCP will route around them but corridors may be distorted.",
    length(bad), paste(bad, collapse = ", ")
  ))

# =============================================================================
# 2. BUILD HABITAT GRAPH (nearest-neighbour edges)
#
# Only route between neighbouring patches — avoids O(n²) pairs for large
# core counts and prevents ecologically meaningless long-distance corridors.
# k = 3 nearest neighbours is a reasonable default; increase for denser graphs.
# =============================================================================
cat("── 2. Building habitat graph ──\n")

K_NEIGHBOURS <- 3

coords_m <- st_coordinates(centroids)   # matrix [n × 2]

# k-NN in Euclidean space (fast, CRS is metric so distances are in metres)
nn <- RANN::nn2(coords_m, k = K_NEIGHBOURS + 1)   # +1 because self is included

edges <- do.call(rbind, lapply(seq_len(nrow(coords_m)), function(i) {
  neighbours <- nn$nn.idx[i, -1]   # exclude self
  data.frame(from = i, to = neighbours)
})) %>%
  # Remove duplicate pairs (i→j and j→i are the same corridor)
  filter(from < to) %>%
  distinct()

cat(sprintf("  %d edges built (k = %d neighbours)\n", nrow(edges), K_NEIGHBOURS))

# =============================================================================
# 3. CUMULATIVE CORRIDOR SURFACE
#
# terra::costDist() computes the accumulated cost distance from one or more
# source cells across the resistance raster. It uses the resistance values
# directly — no manual transition matrix needed.
#
# corridor(i,j) = costDist_from_i + costDist_from_j
# The cell where this sum is minimised lies on the least-cost path.
# =============================================================================
cat("── 3. Computing corridors ──\n")

cum_cost_min <- rast(resistance)
values(cum_cost_min) <- Inf

for (e in seq_len(nrow(edges))) {
  i <- edges$from[e]
  j <- edges$to[e]

  src_i <- cores_vect[i]
  src_j <- cores_vect[j]

  # costDist returns accumulated resistance distance from the source cell(s)
  acc_i <- costDist(resistance, src_i)
  acc_j <- costDist(resistance, src_j)

  corridor_ij <- acc_i + acc_j

  # Running cell-wise minimum — keeps the best (lowest-cost) corridor value
  cum_cost_min <- min(cum_cost_min, corridor_ij, na.rm = TRUE)

  if (e %% 10 == 0 || e == nrow(edges))
    cat(sprintf("    Edge %d / %d\n", e, nrow(edges)))
}

# Unreachable cells (barriers with no path) → NA
cum_cost_min[is.infinite(cum_cost_min)] <- NA

# =============================================================================
# 4. CLIP TO CORRIDOR ENVELOPE
#
# The raw cumulative cost surface covers the entire landscape. Clip it to
# only the pixels that fall within CORRIDOR_THRESHOLD × the least-cost path
# cost for each corridor. This gives a meaningful corridor width rather than
# a ranked version of the full cost surface.
#
# For each edge: LCP cost = minimum value of corridor(i,j)
#                Threshold = LCP cost × CORRIDOR_THRESHOLD
# =============================================================================
cat("── 4. Clipping to corridor envelope ──\n")

# Compute the LCP cost per edge (minimum value of its corridor raster)
# and build a threshold mask
threshold_mask <- rast(resistance)
values(threshold_mask) <- Inf

for (e in seq_len(nrow(edges))) {
  i <- edges$from[e]
  j <- edges$to[e]

  acc_i       <- costDist(resistance, cores_vect[i])
  acc_j       <- costDist(resistance, cores_vect[j])
  corridor_ij <- acc_i + acc_j

  lcp_cost    <- global(corridor_ij, "min", na.rm = TRUE)[[1]]
  cutoff      <- lcp_cost * CORRIDOR_THRESHOLD

  # Pixels below cutoff belong to this corridor's envelope
  corridor_ij[corridor_ij > cutoff] <- Inf
  threshold_mask <- min(threshold_mask, corridor_ij, na.rm = TRUE)
}

# Mask the cumulative surface to the corridor envelope
cum_cost_clipped <- mask(cum_cost_min, threshold_mask < Inf, maskvalue = FALSE)

# =============================================================================
# 5. RECLASSIFY INTO DECILES
#
# Inversion: low cumulative cost = preferred corridor = decile 1 (best)
#            high cumulative cost = poor connectivity = decile 10
# =============================================================================
cat("── 5. Reclassifying ──\n")

reclassify_deciles <- function(r, invert = FALSE) {
  vals          <- values(r, na.rm = TRUE)
  breaks        <- unique(quantile(vals, probs = seq(0, 1, 0.1), na.rm = TRUE))

  if (length(breaks) < 3) {
    warning("Too few unique values for decile classification — returning raw raster.")
    return(r)
  }

  classes <- seq_len(length(breaks) - 1)
  if (invert) classes <- rev(classes)

  rcl <- cbind(
    breaks[-length(breaks)],
    breaks[-1],
    classes
  )

  classify(r, rcl = rcl, include.lowest = TRUE)
}

# Raw surface: decile 1 = lowest cost = best corridor
cum_cost_deciles         <- reclassify_deciles(cum_cost_min,     invert = FALSE)
cum_cost_clipped_deciles <- reclassify_deciles(cum_cost_clipped, invert = FALSE)

# =============================================================================
# 6. SAVE & PLOT
# =============================================================================
cat("── 6. Saving outputs ──\n")

writeRaster(cum_cost_min,
            file.path(OUT_DIR, "cumcost_raw.tif"),             overwrite = TRUE)
writeRaster(cum_cost_deciles,
            file.path(OUT_DIR, "cumcost_deciles.tif"),         overwrite = TRUE)
writeRaster(cum_cost_clipped,
            file.path(OUT_DIR, "cumcost_clipped.tif"),         overwrite = TRUE)
writeRaster(cum_cost_clipped_deciles,
            file.path(OUT_DIR, "cumcost_clipped_deciles.tif"), overwrite = TRUE)

png(file.path(OUT_DIR, "corridor_map.png"),
    width = 1800, height = 1800, res = 200)
par(mfrow = c(1, 2))
plot(cum_cost_deciles,
     col  = rev(hcl.colors(10, "Lajolla")),
     main = "Cumulative Cost Corridors\n(full surface, deciles)",
     axes = FALSE)
plot(cum_cost_clipped_deciles,
     col  = rev(hcl.colors(10, "Lajolla")),
     main = sprintf("Clipped Corridors\n(threshold = %.1f × LCP)", CORRIDOR_THRESHOLD),
     axes = FALSE)
dev.off()

cat("\n── Done ──\n")
cat("  cumcost_raw.tif             — full accumulated cost surface\n")
cat("  cumcost_deciles.tif         — full surface, decile-ranked\n")
cat("  cumcost_clipped.tif         — corridor envelope only\n")
cat("  cumcost_clipped_deciles.tif — corridor envelope, decile-ranked\n")
cat("  corridor_map.png\n")

── 1. Loading inputs ──


Warning message:
st_centroid assumes attributes are constant over geometries 


  121 core patches loaded
── 2. Building habitat graph ──
  187 edges built (k = 3 neighbours)
── 3. Computing corridors ──


: [1m[33mError[39m:[22m
[33m![39m Not compatible with requested type: [type=S4; target=double].

In [24]:
# --------------------------------------------- 4. CUMULATIVE COST CORRIDORS -
cat("\n── 4. Cumulative cost corridors (least-cost paths) ──\n")

library(gdistance)
library(terra)
library(sp)
library(sf)

# 1. Convert resistance to gdistance TransitionLayer
# (We keep gdistance here as it is still the most robust for creating the transition matrix)
resistance_rl <- raster::raster(resistance)

cat("  Calculating transition matrix...\n")
tr <- transition(resistance_rl, transitionFunction = function(x) 1 / mean(x), directions = 8)
tr <- geoCorrection(tr, type = "c") 

# 2. Initialize the running minimum raster (Zero-RAM footprint approach)
# We start with Infinity, so any valid path will overwrite it
cum_cost_min <- rast(resistance)
values(cum_cost_min) <- Inf

crs_wkt <- st_crs(cores_filtered_sf)$wkt

cat(sprintf("  Routing %d edges with on-the-fly aggregation...\n", nrow(edges)))

for (e in seq_len(nrow(edges))) {
  i <- edges$from[e]
  j <- edges$to[e]

  # Create spatial points (using WKT is safer for modern PROJ libraries)
  pt_i <- sp::SpatialPoints(coords[i, , drop = FALSE], proj4string = sp::CRS(crs_wkt))
  pt_j <- sp::SpatialPoints(coords[j, , drop = FALSE], proj4string = sp::CRS(crs_wkt))

  # Calculate Accumulated Cost Surfaces
  acc_i <- rast(accCost(tr, pt_i))
  acc_j <- rast(accCost(tr, pt_j))

  # Corridor = sum of both cost surfaces
  # terra's native addition is massively faster than raster::overlay
  corridor_ij <- acc_i + acc_j

  # Update the running minimum immediately
  cum_cost_min <- min(cum_cost_min, corridor_ij, na.rm = TRUE)

  # CRITICAL: Destroy temporary objects and flush RAM
  rm(acc_i, acc_j, corridor_ij)
  gc()

  if (e %% 10 == 0 || e == nrow(edges)) {
    cat(sprintf("    Edge %d/%d processed\n", e, nrow(edges)))
  }
}

# 3. Clean up the final raster
cat("  Finalizing cumulative cost surface...\n")

# Convert any remaining Infinity values (unreachable areas) to NA
cum_cost_min[is.infinite(cum_cost_min)] <- NA

# 4. Safely Reclassify into deciles (1 = preferred corridor, 10 = barrier)
cat("  Calculating corridor deciles...\n")

# Extract valid pixel values
vals_cc <- values(cum_cost_min, na.rm = TRUE)

# Calculate raw decile breaks
decile_breaks <- quantile(vals_cc, probs = seq(0, 1, 0.1), na.rm = TRUE)

# CRITICAL FIX: Remove duplicate breaks to prevent classify() from crashing
decile_breaks <- unique(decile_breaks)

# Dynamically build the reclassification matrix
rcl_matrix <- cbind(
  decile_breaks[-length(decile_breaks)], # From
  decile_breaks[-1],                     # To
  1:(length(decile_breaks)-1)            # New Class ID
)

# Reclassify the raster safely
cum_cost_deciles <- classify(cum_cost_min, rcl = rcl_matrix, include.lowest = TRUE)

# Save Outputs
writeRaster(cum_cost_min,     file.path(OUT_DIR, "cumcost_raw.tif"),     overwrite = TRUE)
writeRaster(cum_cost_deciles, file.path(OUT_DIR, "cumcost_deciles.tif"), overwrite = TRUE)

cat("  Cumulative cost maps saved successfully.\n")


── 4. Cumulative cost corridors (least-cost paths) ──
  Calculating transition matrix...
  Routing 44 edges with on-the-fly aggregation...
    Edge 10/44 processed
    Edge 20/44 processed
    Edge 30/44 processed
    Edge 40/44 processed
    Edge 44/44 processed
  Finalizing cumulative cost surface...
  Calculating corridor deciles...
  Cumulative cost maps saved successfully.


## Priority Map

In [25]:
# --------------------------------------------- 5. COMBINED PRIORITY MAP -----
# Unify pinch-point deciles + cumulative cost deciles into single priority map
# High priority (decile 10) = high current density AND low cumulative cost

cat("\n── 5. Combined priority map ──\n")

# Align grids
pp_aligned <- resample(current_deciles, cum_cost_deciles, method = "near")

# Invert cumulative cost deciles so high = preferred (low cost → high priority)
cc_inv <- 11 - cum_cost_deciles

# Equal-weight combination (mean), then re-decile
priority_raw <- (pp_aligned + cc_inv) / 2

vals_pr <- values(priority_raw, na.rm = TRUE)
breaks_pr <- quantile(vals_pr, probs = seq(0, 1, 0.1), na.rm = TRUE)
priority_deciles <- classify(priority_raw,
                             rcl = cbind(breaks_pr[-length(breaks_pr)],
                                         breaks_pr[-1], 1:10))

# Set impassable areas (max resistance) to priority 0
max_res <- global(resistance, "max", na.rm = TRUE)[[1]]
priority_deciles[resistance >= max_res] <- 0

writeRaster(priority_deciles, file.path(OUT_DIR, "priority_map_deciles.tif"),
            overwrite = TRUE)
cat("  Priority map saved.\n")





── 5. Combined priority map ──


Warning message:
[+] CRS do not match 
Warning message:
[mask] CRS do not match 


  Priority map saved.


In [26]:
# --------------------------------------------- 6. SUMMARY PLOT ---------------

cat("\n── 6. Generating summary figure ──\n")

rasters_to_plot <- list(
  "Pinch-Point Density (Deciles)"  = current_deciles,
  "Cumulative Cost (Deciles)"      = cum_cost_deciles,
  "Combined Priority"              = priority_deciles
)

png(file.path(OUT_DIR, "connectivity_summary.png"),
    width = 3600, height = 1400, res = 220)
par(mfrow = c(1, 3), mar = c(2, 2, 3, 4))
for (nm in names(rasters_to_plot)) {
  plot(rasters_to_plot[[nm]], main = nm,
       col = hcl.colors(10, "Zissou 1"),
       legend = TRUE, axes = FALSE)
  plot(st_geometry(cores_filtered_sf), add = TRUE,
       col = NA, border = "black", lwd = 0.8)
}
dev.off()

cat("\n✔ Pipeline complete. All outputs in:", OUT_DIR, "\n")
cat("
Output files:
  landscape_graph.png            – visual of graph edges + nodes
  graph_edges.csv                – edge list (from/to patch IDs + distance)
  pinchpoint_current_density.tif – raw Circuitscape current density mosaic
  pinchpoint_deciles.tif         – decile-classified pinch-point map
  cumcost_raw.tif                – raw minimum cumulative cost surface
  cumcost_deciles.tif            – decile-classified cumulative cost map
  priority_map_deciles.tif       – combined priority map (0 = impassable, 10 = highest)
  connectivity_summary.png       – 3-panel summary figure\n")


── 6. Generating summary figure ──

✔ Pipeline complete. All outputs in: ../data/processed/connectivity/ 

Output files:
  landscape_graph.png            – visual of graph edges + nodes
  graph_edges.csv                – edge list (from/to patch IDs + distance)
  pinchpoint_current_density.tif – raw Circuitscape current density mosaic
  pinchpoint_deciles.tif         – decile-classified pinch-point map
  cumcost_raw.tif                – raw minimum cumulative cost surface
  cumcost_deciles.tif            – decile-classified cumulative cost map
  priority_map_deciles.tif       – combined priority map (0 = impassable, 10 = highest)
  connectivity_summary.png       – 3-panel summary figure
